In [1]:
import ee
import os

# ── 1. Authenticate & initialise ─────────────────────────────────────────────
# Run `earthengine authenticate` in your terminal the first time.
ee.Initialize(project="ee-marortpab")   # use the asset's home project

# ── 2. Load the CCDC image ────────────────────────────────────────────────────
asset_id = "projects/ee-marortpab/assets/FAO/uganda/CCDC_asset_nabajuzzi_L5789_2000_2025"
image = ee.Image(asset_id)

# ── 3. Inspect bands (optional but useful) ────────────────────────────────────
info = image.getInfo()
bands = [b["id"] for b in info["bands"]]
print(f"Bands ({len(bands)}): {bands[:10]} ...")
print(f"CRS : {info['bands'][0]['crs']}")

# ── 4. Define export parameters ───────────────────────────────────────────────
OUTPUT_DIR  = "./gee_exports"
EXPORT_NAME = "CCDC_asset_nabajuzzi_L5789_2000_2025"
SCALE       = 30          # Landsat native resolution (metres)
MAX_PIXELS  = 1e13        # raise if the region is very large

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 5a. Export to Google Drive (recommended for large assets) ─────────────────
task_drive = ee.batch.Export.image.toDrive(
    image        = image,
    description  = EXPORT_NAME,
    folder       = "GEE_exports",           # Drive folder (created if absent)
    fileNamePrefix = EXPORT_NAME,
    scale        = SCALE,
    maxPixels    = MAX_PIXELS,
    fileFormat   = "GeoTIFF",
    formatOptions = {"cloudOptimized": True},
)
task_drive.start()
print(f"Drive export started  → task id: {task_drive.id}")

# ── 5b. Export to Cloud Storage (alternative) ─────────────────────────────────
# task_gcs = ee.batch.Export.image.toCloudStorage(
#     image          = image,
#     description    = EXPORT_NAME,
#     bucket         = "your-gcs-bucket",
#     fileNamePrefix = f"gee_exports/{EXPORT_NAME}",
#     scale          = SCALE,
#     maxPixels      = MAX_PIXELS,
#     fileFormat     = "GeoTIFF",
#     formatOptions  = {"cloudOptimized": True},
# )
# task_gcs.start()

# ── 6. Poll task status ────────────────────────────────────────────────────────
import time

def wait_for_task(task, poll_seconds=30):
    """Block until the task completes, printing status updates."""
    while True:
        status = task.status()
        state  = status["state"]
        print(f"  [{state}]", flush=True)
        if state in ("COMPLETED", "FAILED", "CANCELLED"):
            if state == "FAILED":
                print("  Error:", status.get("error_message"))
            return state
        time.sleep(poll_seconds)

print("Waiting for export to finish (may take several minutes) …")
final_state = wait_for_task(task_drive)
print(f"Export finished with state: {final_state}")

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.